In [50]:
import re
import nltk
import string
import spacy
import pandas as pd
import numpy as np
import seaborn as sns
from spellchecker import SpellChecker
from wordcloud import WordCloud
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
from sklearn.decomposition import NMF
from sklearn.pipeline import Pipeline
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('words')

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


import the dataset from the competition

In [51]:
df_prompt = pd.read_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\train_prompts.csv')
df_train_essay = pd.read_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\train_essays.csv')

In [52]:
df_prompt

,prompt_id,prompt_name,instructions,source_text
0,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,1,Does the electoral college work?,Write a letter to your state senator in which ...,# What Is the Electoral College? by the Office...


We can see that there is a data imbalance problem between AI and human essays.

There are 1375 human essays, whereas only 3 AI essays in the dataset.

Thus, the next step is to increase the size of the dataset to fix the problem.

In [53]:
print(df_train_essay[df_train_essay['generated']==0]['generated'].value_counts())
print(df_train_essay[df_train_essay['generated']==1]['generated'].value_counts())

generated
0    1375
Name: count, dtype: int64
generated
1    3
Name: count, dtype: int64


Here, I simply checked the distribution of the prompt_id used in the dataset.

prompt_id 0 has been used 707 times. prompt_id 1 has been used 668 times in the dataset.

There is not so much difference in the dataset. 

We will focus more on the content instead of the prompts.

In [54]:
df_train_essay[df_train_essay['generated'] == 0]['prompt_id'].value_counts()

prompt_id
0    707
1    668
Name: count, dtype: int64

In order to generate the similiar essays as in the competition dataset, I checked the word counts in both AI and human essays. 

Thus, we know how many words should the LLMs produce for the essay.

In [55]:
# Add a column to count the number of words in each text
df_train_essay['word_count'] = df_train_essay['text'].apply(lambda x: len(str(x).split()))

# Group by 'generated' and calculate the average word count
average_word_counts = df_train_essay.groupby('generated')['word_count'].mean().reset_index()

average_word_counts

,generated,word_count
0,0,556.768727
1,1,260.666667


Now we will import the dataset created by other Kaggle contributors for balancing the dataset.

https://www.kaggle.com/code/awsaf49/detect-fake-text-kerasnlp-tf-torch-jax-train

In [56]:
# Load external data
ext_df1 = pd.read_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\extended_dataset\\daigt\\train_drcat_04.csv')
ext_df1

,essay_id,text,label,source,prompt,fold
0,E897534557AF,"In recent years, technology has had a profoun...",1,mistral7binstruct_v2,\nTask: Write an essay discussing the positive...,1
1,DFBA34FFE11D,Should students participate in an extracurricu...,0,persuade_corpus,NaN,2
2,af37ecf5,The electoral college is a symbol of mockery a...,0,train_essays,NaN,5
3,5EC2696BAD78,This is why I think the principle should allow...,0,persuade_corpus,NaN,8
4,llama_70b_v1843,I strongly believe that meditation and mindful...,1,llama_70b_v1,Some schools have implemented meditation and m...,0
...,...,...,...,...,...,...
44201,F7341069C4A4,"""Oh man I didn't make the soccer team!"", yelle...",0,persuade_corpus,NaN,7
44202,AFE6E553DAC2,I believe that using this technology could be ...,0,persuade_corpus,NaN,8
44203,falcon_180b_v1_600,The Face on Mars is a fascinating phenomenon t...,1,falcon_180b_v1,You have read the article 'Unmasking the Face ...,3
44204,A5F84C104693,Texting & Driving\n\nUsing your phone while dr...,0,persuade_corpus,NaN,1


In [57]:
df_train_essay

,id,prompt_id,text,generated,word_count
0,0059830c,0,Cars. Cars have been around since they became ...,0,584
1,005db917,0,Transportation is a large necessity in most co...,0,462
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0,744
3,00940276,0,How often do you ride in a car? Do you drive a...,0,686
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0,871
...,...,...,...,...,...
1373,fe6ff9a5,1,There has been a fuss about the Elector Colleg...,0,430
1374,ff669174,0,Limiting car usage has many advantages. Such a...,0,397
1375,ffa247e0,0,There's a new trend that has been developing f...,0,749
1376,ffc237e9,0,As we all know cars are a big part of our soci...,0,525


In [58]:
# data cleaning
df_train_essay = df_train_essay[['text','generated']]
df_train_essay.rename(columns={'generated':'label'}, inplace=True)

# we took a sample of 4000 from the label 0 and 5300 from the label 1
ext_df1_0 = ext_df1[ext_df1['label'] == 0].sample(4000)
ext_df1_1 = ext_df1[ext_df1['label'] == 1].sample(5300)

# concatenate the dataframes from the external data for the two labels
ext_df2 = pd.concat([ext_df1_0, ext_df1_1])
ext_df3 = pd.concat([df_train_essay, ext_df2])

C:\Users\user\AppData\Local\Temp\ipykernel_19708\796065094.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_essay.rename(columns={'generated':'label'}, inplace=True)


In [59]:
# drop the columns that are not needed
ext_df3.drop(columns=['essay_id','source','prompt','fold'],inplace=True)

In [60]:
# check the distribution of the labels once again
ext_df3['label'].value_counts()

label
0    5375
1    5303
Name: count, dtype: int64

In [61]:
ext_df3.to_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\extended_dataset\\dataset_balanced.csv', index=False)

Build the simple Bag of Word classifier

In [33]:
df_test = pd.read_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\extended_dataset\\arg\\machine-test.csv')
df_test['model'].unique()


df_test_argugpt = pd.read_csv('C:\\Users\\user\\OneDrive - University of Sussex\\Second Semester\\Dissertation\\llm-detect-ai-generated-text\\extended_dataset\\arg\\argugpt.csv')
df_test_argugpt['model'].unique()

array(['text-babbage-001', 'text-curie-001', 'text-davinci-001',
       'text-davinci-002', 'text-davinci-003', 'gpt-3.5-turbo', 'gpt2-xl'],
      dtype=object)

Below is the reuslt of using the simple BoW model to classify the Ai and human essays.

The result shows that the 74 percent of the eassys can be predicted correctly.

Later, we will try to use the BERT model for the same test. 

In [62]:
vectorizer = CountVectorizer()

# Fit and transform the extracted spans to a bag-of-words representation
X = vectorizer.fit_transform(ext_df3['text'])

# Split the data
X_train, X_val, y_binary_train, y_binary_val = train_test_split(
    X, ext_df3['label'], test_size=0.2, random_state=42)

print("Revised data preparation complete. Features and labels ready for model training.")

# Train binary classifier
binary_classifier = LogisticRegression(random_state=42)
binary_classifier.fit(X_train, y_binary_train)

# Evaluate binary classifier
binary_predictions = binary_classifier.predict(X_val)
binary_classification_report = classification_report(y_binary_val, binary_predictions, output_dict=True)

# binary_classification_report
report_df = pd.DataFrame(binary_classification_report).transpose()
print(report_df)

print('==================================================')

X_test = vectorizer.transform(df_test['text'])

df_test['label'] = 1

binary_test_predictions = binary_classifier.predict(X_test)

binary_test_classification_report = classification_report(df_test['label'], binary_test_predictions,output_dict=True)

# binary_test_classification_report
binary_test = pd.DataFrame(binary_test_classification_report).transpose()
print(binary_test)

Revised data preparation complete. Features and labels ready for model training.


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score      support
0              0.982293  0.979554  0.980921  1076.000000
1              0.979304  0.982075  0.980688  1060.000000
accuracy       0.980805  0.980805  0.980805     0.980805
macro avg      0.980798  0.980815  0.980805  2136.000000
weighted avg   0.980809  0.980805  0.980805  2136.000000
              precision    recall  f1-score     support
0              0.000000  0.000000  0.000000    0.000000
1              1.000000  0.742857  0.852459  350.000000
accuracy       0.742857  0.742857  0.742857    0.742857
macro avg      0.500000  0.371429  0.426230  350.000000
weighted avg   1.000000  0.742857  0.852459  350.000000


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

In [62]:
# Initialize spell checker
spell = SpellChecker()

# List of pronouns
pronouns = set([
    'i', 'me', 'my', 'mine', 'myself', 
    'we', 'us', 'our', 'ours', 'ourselves', 
    'you', 'your', 'yours', 'yourself', 'yourselves', 
    'he', 'him', 'his', 'himself', 
    'she', 'her', 'hers', 'herself', 
    'it', 'its', 'itself', 
    'they', 'them', 'their', 'theirs', 'themselves'
])

# Function to check word length
def check_word_length(text):
    words = word_tokenize(text)
    if len(words) == 0:
        return 0
    avg_word_length = np.mean([len(word) for word in words])
    return avg_word_length

# Function to check sentence length
def check_sentence_length(text):
    sentences = sent_tokenize(text)
    if len(sentences) == 0:
        return 0
    avg_sentences = np.mean([len(sentence) for sentence in sentences])
    return avg_sentences

# Function to check pronoun usage
def check_pronouns(text):
    words = word_tokenize(text.lower())
    pronoun_count = sum(1 for word in words if word in pronouns)
    return pronoun_count

# Function to check spelling mistakes
def check_spelling(text):
    words = word_tokenize(text)
    misspelled = spell.unknown(words)
    return len(misspelled)

# Function to check passive voice
def check_passive_voice(text, threshold=3):
    sentences = sent_tokenize(text)
    passive_sentences = 0
    
    for sentence in sentences:
        words = word_tokenize(sentence)
        tagged = nltk.pos_tag(words)
        
        if 'VBN' in [tag for word, tag in tagged] and any(aux in [word for word, tag in tagged] for aux in ['was', 'were', 'is', 'been']):
            passive_sentences += 1
    
    return passive_sentences

# Function to analyze the dataset
def analyze_text(text):
    word_length_flag = check_word_length(text)
    sentence_length_flag = check_sentence_length(text)
    pronoun_flag = check_pronouns(text)
    spelling_flag = check_spelling(text)
    passive_voice_flag = check_passive_voice(text)
    
    return {
        'word_length': word_length_flag,
        'sentence_length': sentence_length_flag,
        'pronouns': pronoun_flag,
        'spelling': spelling_flag,
        'passive_voice': passive_voice_flag
    }

# Analyze each text in the dataset
ext_df3['analysis'] = ext_df3['text'].apply(analyze_text)
ext_df3


,text,label,analysis
0,Cars. Cars have been around since they became ...,0,"{'word_length': 4.117021276595745, 'sentence_l..."
1,Transportation is a large necessity in most co...,0,"{'word_length': 4.32509505703422, 'sentence_le..."
2,"""America's love affair with it's vehicles seem...",0,"{'word_length': 4.390736342042755, 'sentence_l..."
3,How often do you ride in a car? Do you drive a...,0,"{'word_length': 4.156521739130435, 'sentence_l..."
4,Cars are a wonderful thing. They are perhaps o...,0,"{'word_length': 3.9659442724458205, 'sentence_..."
...,...,...,...
26339,Motivation can play a key role in many aspects...,1,"{'word_length': 4.62111801242236, 'sentence_le..."
36449,"Hey, ya'll! So, I'm gonna write about how diff...",1,"{'word_length': 3.450127877237852, 'sentence_l..."
3492,Advantages of Limiting Car Usage\n\nLimiting c...,1,"{'word_length': 4.835640138408304, 'sentence_l..."
13624,What is the greatest accomplishment to be you...,1,"{'word_length': 3.968220338983051, 'sentence_l..."


In [68]:
ext_df3.reset_index(drop=True, inplace=True)

In [69]:
analysis_df = pd.json_normalize(ext_df3['analysis'])
analysis_df

,word_length,sentence_length,pronouns,spelling,passive_voice
0,4.117021,141.913043,21,26,7
1,4.325095,100.296296,11,15,4
2,4.390736,91.187500,33,30,9
3,4.156522,84.361702,53,26,3
4,3.965944,92.920000,68,61,7
...,...,...,...,...,...
10673,4.621118,109.625000,4,1,0
10674,3.450128,92.000000,55,8,0
10675,4.835640,130.800000,9,10,3
10676,3.968220,161.928571,79,5,1


In [70]:
df = ext_df3.drop(columns=['analysis']).join(analysis_df)
df

,text,label,word_length,sentence_length,pronouns,spelling,passive_voice
0,Cars. Cars have been around since they became ...,0,4.117021,141.913043,21,26,7
1,Transportation is a large necessity in most co...,0,4.325095,100.296296,11,15,4
2,"""America's love affair with it's vehicles seem...",0,4.390736,91.187500,33,30,9
3,How often do you ride in a car? Do you drive a...,0,4.156522,84.361702,53,26,3
4,Cars are a wonderful thing. They are perhaps o...,0,3.965944,92.920000,68,61,7
...,...,...,...,...,...,...,...
10673,Motivation can play a key role in many aspects...,1,4.621118,109.625000,4,1,0
10674,"Hey, ya'll! So, I'm gonna write about how diff...",1,3.450128,92.000000,55,8,0
10675,Advantages of Limiting Car Usage\n\nLimiting c...,1,4.835640,130.800000,9,10,3
10676,What is the greatest accomplishment to be you...,1,3.968220,161.928571,79,5,1


Now we have the analyzed the patterns from both Ai and human essays, we will use these features for the BoW model again.

In [71]:
from scipy.sparse import hstack
vectorizer = CountVectorizer()
X_text = vectorizer.fit_transform(df['text'])

# Extract numerical features
X_numeric = df[['word_length', 'sentence_length', 'pronouns', 'spelling', 'passive_voice']].values

# Combine text and numerical features
X = hstack([X_text, X_numeric])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, df['label'], test_size=0.2, random_state=42)

# Train the binary classifier
binary_classifier = LogisticRegression(max_iter=1000, random_state=42)
binary_classifier.fit(X_train, y_train)

# Evaluate the binary classifier
binary_predictions = binary_classifier.predict(X_val)
binary_classification_report = classification_report(y_val, binary_predictions, output_dict=True)

# Print the classification report
report_df = pd.DataFrame(binary_classification_report).transpose()
print(report_df)


              precision    recall  f1-score      support
0              0.977961  0.989777  0.983834  1076.000000
1              0.989494  0.977358  0.983389  1060.000000
accuracy       0.983614  0.983614  0.983614     0.983614
macro avg      0.983728  0.983568  0.983611  2136.000000
weighted avg   0.983684  0.983614  0.983613  2136.000000
